# Approximation algorithms for bin packing (C++)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/combinatorial-optimization/blob/main/notebooks/bin-packing-approximation_cpp.ipynb)

Executable notebook of the chapter “Approximation algorithms for bin packing” (part “Approximation algorithms” of the course *Combinatorial Optimization*): it contains the four approximation algorithms of the chapter for the bin packing problem: Next Fit, First Fit, Best Fit and Worst Fit, written in C++ and run on every example of the chapter and on every instance of the exercises, checking each result against the expected one.

The full chapter — pseudo-code, examples, proofs — is [on the website](https://fabiofurini.github.io/combinatorial-optimization/approximation-algorithms/bin-packing-approximation/). The same notebook is also available in [Python](https://colab.research.google.com/github/fabiofurini/combinatorial-optimization/blob/main/notebooks/bin-packing-approximation_python.ipynb).

**How to run it.** On Colab click the badge and then *Runtime → Run all*, or run the cells one at a time with `Shift+Enter`. Each C++ file is written to the folder `src/` by its `%%writefile` cell; the cell after `Main.cpp` compiles everything with `g++ -std=c++17 -O2` and runs the examples. Nothing to install (Colab already has `g++`).

**Indices.** In the notes items, jobs, cities, machines and bins are numbered from 1, here from 0: the algorithmic logic is unchanged.

## Declarations

The header declares all the functions of the chapter; each function is in its own file, with the same name.

In [1]:
%%writefile src/BPP_heuristics.h
#ifndef BPP_HEURISTICS_HEADER
#define BPP_HEURISTICS_HEADER

// approximation algorithms for the bin packing problem
// items and bins are indexed starting from 0
// every function returns the number of bins used and stores in bin[i] the bin of item i

int NextFit(int n, int c, int *w, int *bin);
int FirstFit(int n, int c, int *w, int *bin);
int BestFit(int n, int c, int *w, int *bin);
int WorstFit(int n, int c, int *w, int *bin);

#endif

Writing src/BPP_heuristics.h


## Next Fit

Only the last opened bin is considered: if the item does not fit, a new bin is opened.

In [2]:
%%writefile src/NextFit.cpp
#include "BPP_heuristics.h"

// items are processed in the input order
// the function returns the number of bins used
// bin[i] stores the bin assigned to item i
// it is assumed that n >= 1

int NextFit(int n, int c, int *w, int *bin)
{
    int nb = 1;   // current bin (number of open bins)
    int rc  = c;   // residual capacity of the current bin

    for (int i = 0; i < n; i++)
    {
        // if item i does not fit into the current bin, open a new bin
        if (w[i] > rc)
        {
            nb++;  // move to the next bin
            rc = c; // reset the residual capacity
        }

        bin[i] = nb - 1; // pack item i into the current bin
        rc -= w[i];       // update the residual capacity
    }

    return nb;
}

Writing src/NextFit.cpp


## First Fit

The item goes into the first opened bin (in opening order) with enough residual capacity; if there is none, a new bin is opened.

In [3]:
%%writefile src/FirstFit.cpp
#include "BPP_heuristics.h"

// items are processed in the input order
// the function returns the number of bins used
// bin[i] stores the bin assigned to item i
// it is assumed that n >= 1

int FirstFit(int n, int c, int *w, int *bin)
{
    int nb = 1;          // number of currently open bins
    int *rc = new int[n]; // residual capacities of bins (at most n bins)
    rc[0] = c;            // residual capacity of the first bin

    for (int i = 0; i < n; i++)
    {
        int s = -1;      // selected bin for item i (-1 means: not found yet)

        // scan bins in the order they were opened and pick the first that fits
        for (int b = 0; b < nb; b++)
        {
            if (w[i] <= rc[b])
            {
                s = b;  // selected bin
                break;
            }
        }

        // if no open bin can accommodate item i, open a new bin
        if (s == -1)
        {
            rc[nb] = c;   // initialize residual capacity of the new bin
            s = nb;     // select the new bin
            nb++;        // increase the number of open bins
        }

        bin[i] = s;     // pack item i into the selected bin
        rc[s] -= w[i];   // update the residual capacity
    }

    delete[] rc;
    return nb;
}

Writing src/FirstFit.cpp


## Best Fit

The item goes into the opened bin with the smallest residual capacity among those in which it fits.

In [4]:
%%writefile src/BestFit.cpp
#include "BPP_heuristics.h"

// items are processed in the input order
// the function returns the number of bins used
// bin[i] stores the bin assigned to item i
// it is assumed that n >= 1

int BestFit(int n, int c, int *w, int *bin)
{
    int nb = 1;          // number of currently open bins
    int *rc = new int[n]; // residual capacities of bins (at most n bins)
    rc[0] = c;            // residual capacity of the first bin

    for (int i = 0; i < n; i++)
    {
        int s = -1;      // selected bin for item i (-1 means: not found yet)
        int minrc = c + 1; // smallest residual capacity found so far

        // among the bins where item i fits, pick the one with the smallest residual capacity
        for (int b = 0; b < nb; b++)
        {
            if (w[i] <= rc[b] && rc[b] < minrc)
            {
                minrc = rc[b];
                s = b;  // selected bin
            }
        }

        // if no open bin can accommodate item i, open a new bin
        if (s == -1)
        {
            rc[nb] = c;   // initialize residual capacity of the new bin
            s = nb;     // select the new bin
            nb++;        // increase the number of open bins
        }

        bin[i] = s;     // pack item i into the selected bin
        rc[s] -= w[i];   // update the residual capacity
    }

    delete[] rc;
    return nb;
}

Writing src/BestFit.cpp


## Worst Fit

The item goes into the opened bin with the largest residual capacity, if it fits there.

In [5]:
%%writefile src/WorstFit.cpp
#include "BPP_heuristics.h"

// items are processed in the input order
// the function returns the number of bins used
// bin[i] stores the bin assigned to item i
// it is assumed that n >= 1

int WorstFit(int n, int c, int *w, int *bin)
{
    int nb = 1;          // number of currently open bins
    int *rc = new int[n]; // residual capacities of bins (at most n bins)
    rc[0] = c;            // residual capacity of the first bin

    for (int i = 0; i < n; i++)
    {
        int s = -1;      // selected bin for item i (-1 means: not found yet)
        int maxrc = -1;   // largest residual capacity found so far

        // among the bins where item i fits, pick the one with the largest residual capacity
        for (int b = 0; b < nb; b++)
        {
            if (w[i] <= rc[b] && rc[b] > maxrc)
            {
                maxrc = rc[b];
                s = b;  // selected bin
            }
        }

        // if no open bin can accommodate item i, open a new bin
        if (s == -1)
        {
            rc[nb] = c;   // initialize residual capacity of the new bin
            s = nb;     // select the new bin
            nb++;        // increase the number of open bins
        }

        bin[i] = s;     // pack item i into the selected bin
        rc[s] -= w[i];   // update the residual capacity
    }

    delete[] rc;
    return nb;
}

Writing src/WorstFit.cpp


## Execution on the examples of the chapter

Every example of the chapter and every instance of the exercises, with the expected result written in the code: each line shows the result found and `OK` when it is the expected one; the last line counts the checks and the errors.

In [6]:
%%writefile src/Main.cpp
#include <cstdio>
#include "BPP_heuristics.h"

// runs every example of the lecture notes and every instance of the exercises,
// and compares the number of bins and the item-bin assignment with the expected ones

static int errors = 0, checks = 0;

static void check(const char* instance, const char* algorithm, int n,
                  int nb, const int* bin, int nb_expected, const int* bin_expected)
{
    checks++;
    bool ok = (nb == nb_expected);
    for (int i = 0; i < n; ++i) if (bin[i] != bin_expected[i]) ok = false;
    if (!ok) errors++;
    printf("%-18s %-9s nb=%2d  bin=(", instance, algorithm, nb);
    for (int i = 0; i < n; ++i) printf("%s%d", i ? " " : "", bin[i]);
    printf(")  %s\n", ok ? "OK" : "<-- EXPECTED A DIFFERENT SOLUTION");
}

int main()
{
    int bin[32];
    { int w[] = {6,5,4,7,3};
      { int e[] = {0,1,1,2,2}; int nb = NextFit(5, 10, w, bin); check("Example (c=10)", "NextFit", 5, nb, bin, 3, e); }
      { int e[] = {0,1,0,2,1}; int nb = FirstFit(5, 10, w, bin); check("Example (c=10)", "FirstFit", 5, nb, bin, 3, e); }
      { int e[] = {0,1,0,2,2}; int nb = BestFit(5, 10, w, bin); check("Example (c=10)", "BestFit", 5, nb, bin, 3, e); }
      { int e[] = {0,1,1,2,0}; int nb = WorstFit(5, 10, w, bin); check("Example (c=10)", "WorstFit", 5, nb, bin, 3, e); }
    }
    { int w[] = {5,5,5,5};
      { int e[] = {0,0,1,1}; int nb = NextFit(4, 10, w, bin); check("Half-full NF ok", "NextFit", 4, nb, bin, 2, e); }
      { int e[] = {0,0,1,1}; int nb = FirstFit(4, 10, w, bin); check("Half-full NF ok", "FirstFit", 4, nb, bin, 2, e); }
      { int e[] = {0,0,1,1}; int nb = BestFit(4, 10, w, bin); check("Half-full NF ok", "BestFit", 4, nb, bin, 2, e); }
      { int e[] = {0,0,1,1}; int nb = WorstFit(4, 10, w, bin); check("Half-full NF ok", "WorstFit", 4, nb, bin, 2, e); }
    }
    { int w[] = {7,7,3,4};
      { int e[] = {0,1,1,2}; int nb = NextFit(4, 10, w, bin); check("Half-full FF", "NextFit", 4, nb, bin, 3, e); }
      { int e[] = {0,1,0,2}; int nb = FirstFit(4, 10, w, bin); check("Half-full FF", "FirstFit", 4, nb, bin, 3, e); }
      { int e[] = {0,1,0,2}; int nb = BestFit(4, 10, w, bin); check("Half-full FF", "BestFit", 4, nb, bin, 3, e); }
      { int e[] = {0,1,0,2}; int nb = WorstFit(4, 10, w, bin); check("Half-full FF", "WorstFit", 4, nb, bin, 3, e); }
    }
    { int w[] = {3,8,3,8};
      { int e[] = {0,1,2,3}; int nb = NextFit(4, 10, w, bin); check("NF not Any Fit", "NextFit", 4, nb, bin, 4, e); }
      { int e[] = {0,1,0,2}; int nb = FirstFit(4, 10, w, bin); check("NF not Any Fit", "FirstFit", 4, nb, bin, 3, e); }
      { int e[] = {0,1,0,2}; int nb = BestFit(4, 10, w, bin); check("NF not Any Fit", "BestFit", 4, nb, bin, 3, e); }
      { int e[] = {0,1,0,2}; int nb = WorstFit(4, 10, w, bin); check("NF not Any Fit", "WorstFit", 4, nb, bin, 3, e); }
    }
    { int w[] = {2,9,2,9};
      { int e[] = {0,1,2,3}; int nb = NextFit(4, 10, w, bin); check("Half-full NF bad", "NextFit", 4, nb, bin, 4, e); }
      { int e[] = {0,1,0,2}; int nb = FirstFit(4, 10, w, bin); check("Half-full NF bad", "FirstFit", 4, nb, bin, 3, e); }
      { int e[] = {0,1,0,2}; int nb = BestFit(4, 10, w, bin); check("Half-full NF bad", "BestFit", 4, nb, bin, 3, e); }
      { int e[] = {0,1,0,2}; int nb = WorstFit(4, 10, w, bin); check("Half-full NF bad", "WorstFit", 4, nb, bin, 3, e); }
    }
    { int w[] = {6,6,4,4};
      { int e[] = {0,1,1,2}; int nb = NextFit(4, 10, w, bin); check("EX_*_1", "NextFit", 4, nb, bin, 3, e); }
      { int e[] = {0,1,0,1}; int nb = FirstFit(4, 10, w, bin); check("EX_*_1", "FirstFit", 4, nb, bin, 2, e); }
      { int e[] = {0,1,0,1}; int nb = BestFit(4, 10, w, bin); check("EX_*_1", "BestFit", 4, nb, bin, 2, e); }
      { int e[] = {0,1,0,1}; int nb = WorstFit(4, 10, w, bin); check("EX_*_1", "WorstFit", 4, nb, bin, 2, e); }
    }
    { int w[] = {7,7,3,3};
      { int e[] = {0,1,1,2}; int nb = NextFit(4, 10, w, bin); check("EX_*_2", "NextFit", 4, nb, bin, 3, e); }
      { int e[] = {0,1,0,1}; int nb = FirstFit(4, 10, w, bin); check("EX_*_2", "FirstFit", 4, nb, bin, 2, e); }
      { int e[] = {0,1,0,1}; int nb = BestFit(4, 10, w, bin); check("EX_*_2", "BestFit", 4, nb, bin, 2, e); }
      { int e[] = {0,1,0,1}; int nb = WorstFit(4, 10, w, bin); check("EX_*_2", "WorstFit", 4, nb, bin, 2, e); }
    }
    { int w[] = {4,7,6,3,5};
      { int e[] = {0,1,2,2,3}; int nb = NextFit(5, 10, w, bin); check("EX_*_3", "NextFit", 5, nb, bin, 4, e); }
      { int e[] = {0,1,0,1,2}; int nb = FirstFit(5, 10, w, bin); check("EX_*_3", "FirstFit", 5, nb, bin, 3, e); }
      { int e[] = {0,1,0,1,2}; int nb = BestFit(5, 10, w, bin); check("EX_*_3", "BestFit", 5, nb, bin, 3, e); }
      { int e[] = {0,1,0,1,2}; int nb = WorstFit(5, 10, w, bin); check("EX_*_3", "WorstFit", 5, nb, bin, 3, e); }
    }
    { int w[] = {4,2,4,2,4,2,4,2};
      { int e[] = {0,0,1,1,2,2,3,3}; int nb = NextFit(8, 8, w, bin); check("worst case n=8", "NextFit", 8, nb, bin, 4, e); }
      { int e[] = {0,0,1,0,1,2,2,2}; int nb = FirstFit(8, 8, w, bin); check("worst case n=8", "FirstFit", 8, nb, bin, 3, e); }
      { int e[] = {0,0,1,0,1,2,2,2}; int nb = BestFit(8, 8, w, bin); check("worst case n=8", "BestFit", 8, nb, bin, 3, e); }
      { int e[] = {0,0,1,1,2,2,3,3}; int nb = WorstFit(8, 8, w, bin); check("worst case n=8", "WorstFit", 8, nb, bin, 4, e); }
    }
    { int w[] = {6,2,6,2,6,2,6,2,6,2,6,2};
      { int e[] = {0,0,1,1,2,2,3,3,4,4,5,5}; int nb = NextFit(12, 12, w, bin); check("worst case n=12", "NextFit", 12, nb, bin, 6, e); }
      { int e[] = {0,0,1,0,1,0,2,2,3,2,3,2}; int nb = FirstFit(12, 12, w, bin); check("worst case n=12", "FirstFit", 12, nb, bin, 4, e); }
      { int e[] = {0,0,1,0,1,0,2,2,3,2,3,2}; int nb = BestFit(12, 12, w, bin); check("worst case n=12", "BestFit", 12, nb, bin, 4, e); }
      { int e[] = {0,0,1,1,2,2,3,3,4,4,5,5}; int nb = WorstFit(12, 12, w, bin); check("worst case n=12", "WorstFit", 12, nb, bin, 6, e); }
    }
    { int w[] = {8,2,8,2,8,2,8,2,8,2,8,2,8,2,8,2};
      { int e[] = {0,0,1,1,2,2,3,3,4,4,5,5,6,6,7,7}; int nb = NextFit(16, 16, w, bin); check("worst case n=16", "NextFit", 16, nb, bin, 8, e); }
      { int e[] = {0,0,1,0,1,0,2,0,2,3,3,3,4,3,4,3}; int nb = FirstFit(16, 16, w, bin); check("worst case n=16", "FirstFit", 16, nb, bin, 5, e); }
      { int e[] = {0,0,1,0,1,0,2,0,2,3,3,3,4,3,4,3}; int nb = BestFit(16, 16, w, bin); check("worst case n=16", "BestFit", 16, nb, bin, 5, e); }
      { int e[] = {0,0,1,1,2,2,3,3,4,4,5,5,6,6,7,7}; int nb = WorstFit(16, 16, w, bin); check("worst case n=16", "WorstFit", 16, nb, bin, 8, e); }
    }

    printf("\n%d checks, %d errors\n", checks, errors);
    return errors == 0 ? 0 : 1;
}

Writing src/Main.cpp


In [7]:
!g++ -std=c++17 -O2 -o BPP_HEUR src/*.cpp && ./BPP_HEUR

Example (c=10)     NextFit   nb= 3  bin=(0 1 1 2 2)  OK
Example (c=10)     FirstFit  nb= 3  bin=(0 1 0 2 1)  OK
Example (c=10)     BestFit   nb= 3  bin=(0 1 0 2 2)  OK
Example (c=10)     WorstFit  nb= 3  bin=(0 1 1 2 0)  OK
Half-full NF ok    NextFit   nb= 2  bin=(0 0 1 1)  OK
Half-full NF ok    FirstFit  nb= 2  bin=(0 0 1 1)  OK
Half-full NF ok    BestFit   nb= 2  bin=(0 0 1 1)  OK
Half-full NF ok    WorstFit  nb= 2  bin=(0 0 1 1)  OK
Half-full FF       NextFit   nb= 3  bin=(0 1 1 2)  OK
Half-full FF       FirstFit  nb= 3  bin=(0 1 0 2)  OK
Half-full FF       BestFit   nb= 3  bin=(0 1 0 2)  OK
Half-full FF       WorstFit  nb= 3  bin=(0 1 0 2)  OK
NF not Any Fit     NextFit   nb= 4  bin=(0 1 2 3)  OK
NF not Any Fit     FirstFit  nb= 3  bin=(0 1 0 2)  OK
NF not Any Fit     BestFit   nb= 3  bin=(0 1 0 2)  OK
NF not Any Fit     WorstFit  nb= 3  bin=(0 1 0 2)  OK
Half-full NF bad   NextFit   nb= 4  bin=(0 1 2 3)  OK
Half-full NF bad   FirstFit  nb= 3  bin=(0 1 0 2)  OK
Half-full NF bad   B

## Try it yourself

Change the data in the cell below and run it again.

In [8]:
%%writefile try.cpp
#include <cstdio>
#include "src/BPP_heuristics.h"

// Try it yourself: change the data below, compile and run again
int c = 10;                     // bin capacity
int w[] = {6, 5, 4, 7, 3};      // item weights (each at most c)
int n = sizeof(w) / sizeof(w[0]);

int main()
{
    const char* names[] = {"Next Fit", "First Fit", "Best Fit", "Worst Fit"};
    int* bin = new int[n];

    printf("Instance: c = %d, n = %d items\n", c, n);
    for (int a = 0; a < 4; ++a)
    {
        int nb = 0;
        if (a == 0) nb = NextFit(n, c, w, bin);
        if (a == 1) nb = FirstFit(n, c, w, bin);
        if (a == 2) nb = BestFit(n, c, w, bin);
        if (a == 3) nb = WorstFit(n, c, w, bin);
        printf("%-9s : %d bins\n", names[a], nb);
        for (int b = 0; b < nb; ++b)
        {
            printf("    bin %d: items", b);
            int load = 0;
            for (int i = 0; i < n; ++i)
            {
                if (bin[i] == b)
                {
                    printf(" %d", i);
                    load += w[i];
                }
            }
            printf("  (load %d)\n", load);
        }
    }

    delete[] bin;
    return 0;
}

Writing try.cpp


In [9]:
!g++ -std=c++17 -O2 -o try try.cpp $(ls src/*.cpp | grep -v Main.cpp) && ./try

Instance: c = 10, n = 5 items
Next Fit  : 3 bins
    bin 0: items 0  (load 6)
    bin 1: items 1 2  (load 9)
    bin 2: items 3 4  (load 10)
First Fit : 3 bins
    bin 0: items 0 2  (load 10)
    bin 1: items 1 4  (load 8)
    bin 2: items 3  (load 7)
Best Fit  : 3 bins
    bin 0: items 0 2  (load 10)
    bin 1: items 1  (load 5)
    bin 2: items 3 4  (load 10)
Worst Fit : 3 bins
    bin 0: items 0 4  (load 9)
    bin 1: items 1 2  (load 9)
    bin 2: items 3  (load 7)


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/combinatorial-optimization/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/combinatorial-optimization/blob/main/LICENSE-CODE).